# 11 · Fit the control guides' effects

The same fit as notebook 10, run over the control guides instead of the
knockout guides. Control guides should show no effect, so the distribution of
these coefficients is the null the knockout effects are read against.

**Reads** `par_save_filename_7`.
**Writes** `par_control_coefs_file` and `par_control_pvals_file`.

Ordinary least squares is used here rather than the negative binomial of
notebook 10: this is a reference distribution for comparison, not an effect
size that gets reported per guide.

:::{note}
This fit takes a long time across all control guides, so its result is provided
rather than regenerated: `Control_coefs.csv` and `Control_pValues.csv` are
downloaded separately, as described in `TextFiles/README.md`.

Running this notebook writes its own copies under `outputs/` and leaves the
downloaded files untouched.
:::


## Setup

In [ ]:
from libraries import *
from parameters import *
from pathlib import Path

os.chdir(projectDir)
import statsmodels.api as sm

Path(par_control_lm_dir).mkdir(parents=True, exist_ok=True)
Path(par_control_coefs_file).parent.mkdir(parents=True, exist_ok=True)

In [ ]:
adata = sc.read(par_save_filename_7)

control_guides = [
    g for g in adata.uns["feature_barcode_names_filtered"]
    if g.startswith((par_not_target_control_prefix, par_nongene_site_control_prefix))
]
print(f"control guides: {len(control_guides)}")

covariates = adata.obs[["n_genes", "mt_frac", "leiden"]]
covariates = covariates.join(pd.get_dummies(covariates.leiden)).drop(columns=["leiden"])
design = adata.obs[control_guides].join(covariates).astype(float)

expression = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
gene_names = list(adata.var_names)
print(f"design: {design.shape[0]} cells x {design.shape[1]} covariates")

## Fit one model per gene, in blocks

In [ ]:
n_genes = len(gene_names)
for start in range(0, n_genes, par_gene_block_size):
    stop = min(start + par_gene_block_size, n_genes)
    out = f"{par_control_lm_dir}/coefs_{start}_{stop}_ControlGuide.csv"
    if Path(out).exists():
        continue

    coefs, pvals = pd.DataFrame(), pd.DataFrame()
    for j in range(start, stop):
        fit = sm.OLS(expression[:, j], np.array(design)).fit()
        table = fit.summary2().tables[1]
        coefs[gene_names[j]] = table["Coef."]
        pvals[gene_names[j]] = table["P>|t|"]

    coefs.index = design.columns
    pvals.index = design.columns
    coefs.to_csv(out)
    pvals.to_csv(f"{par_control_lm_dir}/pvalues_{start}_{stop}_ControlGuide.csv")
    print(f"  genes {start}-{stop} done")

print("all blocks fitted")

## Assemble and write

In [ ]:
def load_blocks(prefix):
    frames = []
    for start in range(0, n_genes, par_gene_block_size):
        stop = min(start + par_gene_block_size, n_genes)
        frames.append(pd.read_csv(
            f"{par_control_lm_dir}/{prefix}_{start}_{stop}_ControlGuide.csv", index_col=0))
    return pd.concat(frames, axis=1)

coefs = load_blocks("coefs").loc[control_guides]
pvals = load_blocks("pvalues").loc[control_guides]

coefs.to_csv(par_control_coefs_file)
pvals.to_csv(par_control_pvals_file)
print(f"written: {par_control_coefs_file}  ({coefs.shape[0]} guides x {coefs.shape[1]} genes)")
print(f"written: {par_control_pvals_file}")